In [1]:
import mlflow
import dagshub

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow")

In [3]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='mental-health-score-predictor', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/mental-health-score-predictor"

Repository Aayush10671/mental-health-score-predictor initialized!

🏃 View run vaunted-mare-561 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/0/runs/9256e234e5d44b35b0894e78f22e2422
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/0


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso,LogisticRegression

In [5]:
mlflow.set_experiment("experiment-2----finding out best model")

<Experiment: artifact_location='mlflow-artifacts:/820ff6f14ca74ef6b15c38ba621e22a8', creation_time=1785602611430, experiment_id='2', last_update_time=1785602611430, lifecycle_stage='active', name='experiment-2----finding out best model', tags={}, workspace='default'>

In [6]:
df = pd.read_csv('../data/raw/dataset.csv')

In [7]:
num_feature = df.select_dtypes(include = 'number')
len(num_feature.columns)

7

In [8]:
df.shape

(5000, 13)

In [9]:
Q1 = num_feature.quantile(0.25)
Q3 = num_feature.quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = ((num_feature < lower_limit) | (num_feature > upper_limit)).sum()
outliers


Age                         0
Avg_Daily_Usage_Hours       0
Daily_Unlocks               0
Study_Hours                 2
Physical_Activity_Hours    22
Sleep_Hours_Per_Night       0
Mental_Health_Score         0
dtype: int64

In [10]:
df.drop_duplicates(inplace=True)

df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)

In [11]:
top_countries = df['Country'].value_counts().head(11)
def group_countries(country):
    if country in top_countries.index:
        return country
    else:
        return 'Other'

In [12]:
df['Grouped_Country'] = df['Country'].apply(group_countries)

In [13]:
df = df[~df.isin(outliers).any(axis=1)]

In [14]:
skewed_col = ['Study_Hours']
other_numric_col = ['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Physical_Activity_Hours' , 'Daily_Unlocks']
ordinal_col = ['Stress_Level']

normal_col = ['Gender','Academic_Level','Most_Used_Platform','Grouped_Country' , 'Purpose_Of_Use']  
features_col = skewed_col + other_numric_col + ordinal_col + normal_col

X = df[features_col]
y = df['Mental_Health_Score']

In [15]:
df.shape

(4998, 14)

In [16]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder,OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

skew_pipeline = Pipeline(steps=[('log_transformer', FunctionTransformer(np.log1p, validate=True)), ('scaler', StandardScaler())])
                                
plain_pipeline = Pipeline(steps=[('scaler', StandardScaler())])

ordinal_pipeline = Pipeline(steps=[('ordinal_encoder', OrdinalEncoder(categories=[['Low', 'Medium', 'High' , 'Very High']]))])

nominal_pipeline = Pipeline(steps=[('onehot_encoder', OneHotEncoder(drop='first' , handle_unknown='ignore'))])


preprocessor = ColumnTransformer(transformers=[
    ('skew', skew_pipeline, skewed_col),
    ('plain', plain_pipeline, other_numric_col),
    ('ordinal', ordinal_pipeline, ordinal_col),
    ('nominal', nominal_pipeline, normal_col)
])

In [23]:
algos = {
    "XGboost" : XGBRegressor(),
    "RandomForestRegressor" : RandomForestRegressor(),
    "LinearRegression" : LinearRegression(),
    "lightGBM" : LGBMRegressor(),
}

In [25]:
with mlflow.start_run(run_name="All experiments") as parent_run:
    for algo_name, algo in algos.items():

        with mlflow.start_run(run_name=algo_name, nested=True) as child_run:

            mlflow.log_param("model_name", algo_name)
            mlflow.log_param("test_size", 0.3)

            model = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", algo)
                ]
            )

            X_train, X_test, y_train, y_test = train_test_split(
                X, y,
                test_size=0.3,
                random_state=42
            )

            model.fit(X_train, y_train)

            estimator = model.named_steps["model"]

            if algo_name == "XGboost":
                mlflow.log_param("n_estimators", estimator.n_estimators)
                mlflow.log_param("max_depth", estimator.max_depth)
                mlflow.log_param("learning_rate", estimator.learning_rate)

            elif algo_name == "RandomForestRegressor":
                mlflow.log_param("n_estimators", estimator.n_estimators)
                mlflow.log_param("max_depth", estimator.max_depth)
                mlflow.log_param("min_samples_split", estimator.min_samples_split)

            elif algo_name == "lightGBM":
                mlflow.log_param("n_estimators", estimator.n_estimators)
                mlflow.log_param("max_depth", estimator.max_depth)
                mlflow.log_param("learning_rate", estimator.learning_rate)

            elif algo_name == "LinearRegression":
                mlflow.log_param("fit_intercept", estimator.fit_intercept)

            y_pred = model.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            mlflow.log_metric("mse", mse)
            mlflow.log_metric("r2", r2)

🏃 View run XGboost at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2/runs/3e8fe3041e80493889899af21836b806
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2
🏃 View run RandomForestRegressor at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2/runs/d22b536fd0e44ffa8bdfb6e4b06077ba
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2
🏃 View run LinearRegression at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2/runs/fcd21bf6f24f4cd8ad517bc7900e2892
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003022 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


🏃 View run lightGBM at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2/runs/3a710facefbb47349591682271478890
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2
🏃 View run All experiments at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2/runs/e976e2efa46f45ac93382ad4b6b050a0
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/2
